# WS 01 — Bộ gán nhãn văn bản: Bio_ClinicalBERT fine-tune trên MedNLI

Notebook Google Colab này tạo **teacher NLI 3 lớp** để sinh nhãn yếu ở giai đoạn 3. Teacher nhận cặp *(premise, hypothesis)* và trả về `entailment`, `neutral`, `contradiction`.

- Bật **Runtime → Change runtime type → GPU** trước khi chạy.
- MedNLI và checkpoint tạo ra là dữ liệu/tài nguyên nội bộ: **không công khai hoặc tải lên Hugging Face**.
- Test MedNLI chỉ chạy một lần sau khi chốt checkpoint theo **dev accuracy**, không dùng để chọn model hoặc điều chỉnh siêu tham số.
- Sau khi hoàn tất, Trí cần duyệt `metrics.json` và `sanity_derm.csv` trước khi sang giai đoạn 3.

## 1. Chuẩn bị Python và repository

Notebook chạy trên Colab và dùng bản repository ở `/content/RAD-colab`. Đặt `BRANCH = "WS"` khi nhánh đã được tạo từ `origin/P1`; trong lúc chuẩn bị có thể thay bằng nhánh làm việc hiện tại của Trí.

In [ ]:
import subprocess
import sys
from pathlib import Path

print("Python:", sys.version)
REPO_DIR = Path("/content/RAD-colab")
REPO_URL = "https://github.com/trisle0805/RAD-colab.git"
BRANCH = "WS"  # Đổi nếu tên branch làm việc khác.

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

assert (REPO_DIR / "main_rad.py").is_file(), "Sai repository: không thấy main_rad.py"
GIT_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
print(f"✅ Repository: {REPO_DIR}\n✅ Branch: {BRANCH}\n✅ Commit: {GIT_COMMIT}")

## 2. Cài dependency và xác nhận GPU CUDA

Không cài lại `torch` để giữ phiên bản CUDA do Colab cung cấp.

In [ ]:
%pip install -q -U "transformers>=4.40" "datasets>=2.18" "accelerate>=0.28" "evaluate>=0.4" scikit-learn

import importlib
from importlib.metadata import version
import torch

for module, package in {
    "transformers": "transformers", "datasets": "datasets", "accelerate": "accelerate",
    "evaluate": "evaluate", "sklearn": "scikit-learn", "numpy": "numpy", "pandas": "pandas",
}.items():
    importlib.import_module(module)
    print(f"✅ {package}: {version(package)}")

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime → Change runtime type → GPU"
print("GPU:", torch.cuda.get_device_name(0))

## 3. Mount Drive, thiết lập experiment và xác thực MedNLI

Để `RUN_ID = None` khi tạo run mới; để resume, dán đúng timestamp đã tạo. Output luôn ở `Proposition-outputs/<EXPERIMENT_NAME>/<RUN_ID>/`.

In [ ]:
import hashlib
import json
import random
from datetime import datetime, timezone

import numpy as np
from google.colab import drive

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

MEDNLI_DIR = Path("/content/drive/MyDrive/Cao học/MedNLI")
MEDNLI_PATHS = {
    "train": MEDNLI_DIR / "mli_train_v1.jsonl",
    "dev": MEDNLI_DIR / "mli_dev_v1.jsonl",
    "test": MEDNLI_DIR / "mli_test_v1.jsonl",
}
BERT_MODEL_ID = "emilyalsentzer/Bio_ClinicalBERT"
EXPERIMENT_NAME = "weak-teacher-text-bioclinicalbert-mednli"
RUNS_ROOT = Path("/content/drive/MyDrive/Cao học/Proposition-outputs") / EXPERIMENT_NAME
RUN_ID = None  # Dán timestamp tại đây khi resume/xem lại một run cũ.

for split, path in MEDNLI_PATHS.items():
    assert path.is_file(), f"Không tìm thấy MedNLI {split}: {path}"

RUNS_ROOT.mkdir(parents=True, exist_ok=True)
if RUN_ID is None:
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    RUN_DIR = RUNS_ROOT / RUN_ID
    RUN_DIR.mkdir(exist_ok=False)
    print(f"✅ Tạo run mới: {RUN_ID} (UTC)")
else:
    RUN_DIR = RUNS_ROOT / RUN_ID
    assert RUN_DIR.is_dir(), f"Không tìm thấy run để resume: {RUN_DIR}"
    print(f"✅ Dùng lại run: {RUN_ID}")

for directory in ("model", "trainer_checkpoints"):
    (RUN_DIR / directory).mkdir(exist_ok=True)
print("Run directory:", RUN_DIR)

## 4. Nạp MedNLI, cố định nhãn và kiểm tra dữ liệu

Quy ước nhãn là bất biến: `entailment=0`, `neutral=1`, `contradiction=2`.

In [ ]:
from collections import Counter
from datasets import load_dataset
import pandas as pd

LABEL2ID = {"entailment": 0, "neutral": 1, "contradiction": 2}
ID2LABEL = {value: key for key, value in LABEL2ID.items()}
LABEL_MAP_PATH = RUN_DIR / "label_map.json"
if LABEL_MAP_PATH.is_file():
    assert json.loads(LABEL_MAP_PATH.read_text(encoding="utf-8")) == LABEL2ID, "label_map.json không đúng quy ước cố định."
else:
    LABEL_MAP_PATH.write_text(json.dumps(LABEL2ID, indent=2), encoding="utf-8")

raw_datasets = load_dataset("json", data_files={key: str(value) for key, value in MEDNLI_PATHS.items()})
for split, dataset in raw_datasets.items():
    required = {"sentence1", "sentence2", "gold_label"}
    assert required.issubset(dataset.column_names), f"{split} thiếu trường: {required - set(dataset.column_names)}"
    labels = dataset["gold_label"]
    unexpected = set(labels) - set(LABEL2ID)
    assert not unexpected, f"{split} có nhãn không hợp lệ: {unexpected}"
    print(f"{split:5}: {len(dataset):5} cặp | {dict(Counter(labels))}")

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

SOURCE_SHA256 = {split: sha256_file(path) for split, path in MEDNLI_PATHS.items()}
print("SHA-256 MedNLI:")
for split, digest in SOURCE_SHA256.items():
    print(f"  {split}: {digest}")

## 5. Token hoá cặp NLI và tạo cấu hình huấn luyện

Mỗi input là `(sentence1, sentence2)` với `max_length=128`. Checkpoint được chọn **chỉ theo accuracy trên dev**.

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

MAX_LENGTH = 128
TRAIN_CONFIG = {
    "learning_rate": 2e-5,
    "per_device_train_batch_size": 32,
    "per_device_eval_batch_size": 32,
    "num_train_epochs": 4,
    "warmup_ratio": 0.1,
    "weight_decay": 0.01,
    "seed": SEED,
    "max_length": MAX_LENGTH,
}
CONFIG_PATH = RUN_DIR / "train_config.json"
if CONFIG_PATH.is_file():
    saved_config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    assert saved_config == TRAIN_CONFIG, "Cấu hình khác với run đang resume. Hãy tạo RUN_ID mới."
else:
    CONFIG_PATH.write_text(json.dumps(TRAIN_CONFIG, indent=2), encoding="utf-8")

tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_ID)
def tokenize(batch):
    return tokenizer(batch["sentence1"], batch["sentence2"], truncation=True, max_length=MAX_LENGTH)
def map_label(example):
    example["labels"] = LABEL2ID[example["gold_label"]]
    return example

tokenized = raw_datasets.map(tokenize, batched=True, desc="Tokenizing")
tokenized = tokenized.map(map_label, desc="Mapping fixed labels")
keep_columns = {"input_ids", "attention_mask", "token_type_ids", "labels"}
for split in tokenized:
    tokenized[split] = tokenized[split].remove_columns([name for name in tokenized[split].column_names if name not in keep_columns])

model = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL_ID, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID,
)
print("✅ Model loaded; classification head mới khởi tạo là bình thường.")

def compute_metrics(prediction):
    logits, labels = prediction
    predicted = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predicted),
        "macro_f1": f1_score(labels, predicted, average="macro", zero_division=0),
    }

training_args = TrainingArguments(
    output_dir=str(RUN_DIR / "trainer_checkpoints"),
    learning_rate=TRAIN_CONFIG["learning_rate"],
    per_device_train_batch_size=TRAIN_CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=TRAIN_CONFIG["per_device_eval_batch_size"],
    num_train_epochs=TRAIN_CONFIG["num_train_epochs"],
    warmup_ratio=TRAIN_CONFIG["warmup_ratio"],
    weight_decay=TRAIN_CONFIG["weight_decay"],
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    seed=SEED,
    data_seed=SEED,
    report_to="none",
    logging_strategy="epoch",
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["dev"],
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

## 6. Fine-tune theo dev accuracy

Chạy ô này để train hoặc tiếp tục từ checkpoint gần nhất. `Trainer` đánh giá sau từng epoch, và chỉ nạp lại checkpoint tốt nhất theo `eval_accuracy` trên dev.

In [ ]:
from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = get_last_checkpoint(str(RUN_DIR / "trainer_checkpoints"))
if last_checkpoint:
    print("✅ Resume từ:", last_checkpoint)
else:
    print("ℹ️ Bắt đầu fine-tune mới từ Bio_ClinicalBERT.")

train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
trainer.save_model(str(RUN_DIR / "model"))
tokenizer.save_pretrained(str(RUN_DIR / "model"))

best_checkpoint = trainer.state.best_model_checkpoint
assert best_checkpoint is not None, "Không xác định được checkpoint tốt nhất theo dev accuracy."
print("✅ Best checkpoint theo dev accuracy:", best_checkpoint)
print("✅ Đã lưu teacher + tokenizer:", RUN_DIR / "model")

## 7. Đánh giá test một lần và thử nhanh câu da liễu

Chỉ chạy sau khi fine-tune đã hoàn tất. Không dùng kết quả test để chỉnh model hay cấu hình.

In [ ]:
# Test chỉ thực hiện sau khi checkpoint đã được chốt bởi dev accuracy.
# Không chạy lại ô này với RUN_ID đã có metrics.json: test phải diễn ra đúng một lần.
METRICS_PATH = RUN_DIR / "metrics.json"
assert not METRICS_PATH.exists(), (
    f"Test đã được ghi cho run này: {METRICS_PATH}. Không được chạy test lần hai; "
    "hãy chỉ mở file kết quả hoặc tạo RUN_ID mới cho thí nghiệm mới."
)

test_output = trainer.predict(tokenized["test"], metric_key_prefix="test")
test_predictions = np.argmax(test_output.predictions, axis=-1)
test_labels = test_output.label_ids

per_class_f1 = f1_score(test_labels, test_predictions, labels=[0, 1, 2], average=None, zero_division=0)
test_metrics = {
    "accuracy": float(accuracy_score(test_labels, test_predictions)),
    "macro_f1": float(f1_score(test_labels, test_predictions, average="macro", zero_division=0)),
    "per_class_f1": {ID2LABEL[index]: float(score) for index, score in enumerate(per_class_f1)},
    "confusion_matrix": confusion_matrix(test_labels, test_predictions, labels=[0, 1, 2]).tolist(),
}
print(json.dumps(test_metrics, indent=2))

sanity_cases = [
    ("Multiple sharply demarcated red plaques with silvery scales.", "well-demarcated erythematous plaques.", "entailment"),
    ("Erythematous plaques without any scale.", "scaly plaques.", "contradiction"),
    ("Plaques with silvery scales on the elbows.", "nail pitting.", "neutral"),
    ("Ill-defined red patches on the cheeks.", "well-demarcated patches.", "contradiction"),
    ("Depigmented white patches with clear borders on the hands.", "chalk-white macules or patches.", "entailment"),
    ("Lesions on the face.", "lesions on the face, chest and back.", "record only"),
]
model.eval()
sanity_rows = []
for index, (premise, description, expected) in enumerate(sanity_cases, start=1):
    hypothesis = f"The patient has {description}"
    inputs = tokenizer(premise, hypothesis, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(model.device)
    with torch.no_grad():
        probabilities = torch.softmax(model(**inputs).logits, dim=-1)[0].detach().cpu().numpy()
    predicted_id = int(np.argmax(probabilities))
    sanity_rows.append({
        "case": index, "premise": premise, "hypothesis": hypothesis, "expected": expected,
        "predicted": ID2LABEL[predicted_id],
        **{f"p_{ID2LABEL[label_id]}": float(probabilities[label_id]) for label_id in range(3)},
    })

sanity_frame = pd.DataFrame(sanity_rows)
SANITY_PATH = RUN_DIR / "sanity_derm.csv"
sanity_frame.to_csv(SANITY_PATH, index=False)
display(sanity_frame)
for row in sanity_rows[:3]:
    assert row["predicted"] == row["expected"], f"Sanity case {row['case']} sai: {row['predicted']} != {row['expected']}"
print("✅ Ba sanity cases bắt buộc đầu tiên đạt.")

## 8. Ghi metrics, metadata và kiểm tra artifact

Ô này đóng gói kết quả tái lập và kiểm tra đủ file cần Trí duyệt.

In [ ]:
from importlib.metadata import version

log_history = trainer.state.log_history
dev_history = [entry for entry in log_history if "eval_accuracy" in entry]
metrics_payload = {
    "selection_metric": "dev_accuracy",
    "best_model_checkpoint": best_checkpoint,
    "best_metric": trainer.state.best_metric,
    "dev_by_epoch": dev_history,
    "test": test_metrics,
}
METRICS_PATH = RUN_DIR / "metrics.json"
METRICS_PATH.write_text(json.dumps(metrics_payload, ensure_ascii=False, indent=2), encoding="utf-8")

meta_payload = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "git_commit": GIT_COMMIT,
    "seed": SEED,
    "model_id": BERT_MODEL_ID,
    "label_map": LABEL2ID,
    "training_config": TRAIN_CONFIG,
    "source_files": {split: {"path": str(path), "sha256": SOURCE_SHA256[split]} for split, path in MEDNLI_PATHS.items()},
    "library_versions": {
        name: version(name) for name in ["transformers", "datasets", "accelerate", "evaluate", "scikit-learn", "numpy", "pandas"]
    } | {"torch": torch.__version__},
}
META_PATH = RUN_DIR / "meta.json"
META_PATH.write_text(json.dumps(meta_payload, ensure_ascii=False, indent=2), encoding="utf-8")

required_paths = [RUN_DIR / "model" / "config.json", LABEL_MAP_PATH, METRICS_PATH, SANITY_PATH, META_PATH]
for path in required_paths:
    assert path.is_file(), f"Thiếu artifact: {path}"
    print("✅", path.relative_to(RUN_DIR))

print("\nRun hoàn tất:", RUN_DIR)
print("Trí cần duyệt metrics.json và sanity_derm.csv trước giai đoạn 3.")